# 🔢 PyTorch Handwritten Digit Recognition

A from-scratch 6-layer MLP that learns to classify handwritten digits (0–9) from the MNIST dataset.

**Pipeline:** download CSV → preprocess → wrap in `Dataset`/`DataLoader` → train MLP → evaluate accuracy.


## 0 · Planning

0. MNIST?
1. Installing packages and defining DEVICE
2. Downloading and loading data
3. Datasets and DataLoaders
4. Modeling
5. Training loop
6. Evaluation loop


### About MNIST

- Contains handwritten digits from 0 to 9.
- Total images: 70,000 (60k train + 10k test)
- Image size: 28 × 28 pixels (grayscale)
- Color channels: 1 (since grayscale)
- Classes: 10 (digits 0 through 9)

- Each image (X) → a 2D tensor (28×28), or flattened into a 1D vector of 784 features.
- Label (y) is a single integer representing the digit.


## 1 · Setup & imports


In [ ]:
import os
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

%matplotlib inline


In [ ]:
# Pick GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 2 · Download and explore the data


In [ ]:
csv_path = 'mnist_train.csv'

# Download the file if it doesn't already exist
if not os.path.exists(csv_path):
    print('Downloading MNIST training dataset...')
    url = 'https://pjreddie.com/media/files/mnist_train.csv'
    # Custom User-Agent to prevent 403 Forbidden
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(url, csv_path)
    print('Download finished!')

data = pd.read_csv(csv_path)


In [ ]:
data.head()

In [ ]:
data.shape   # expect (60000, 785): 1 label col + 784 pixel cols


In [ ]:
# Visualize the first 2 digits
for i in range(2):
    feat = data.iloc[i, 1:].to_numpy(dtype=np.float32).reshape(28, 28)
    plt.imshow(feat, cmap='gray')
    plt.title(f'label = {data.iloc[i, 0]}')
    plt.axis('off')
    plt.show()


## 3 · Preprocessing

1. Split features (pixels) and labels (digit).
2. Train/test split (80/20).
3. Normalize pixels from 0–255 → 0.0–1.0.
4. Convert pandas DataFrames to NumPy arrays (PyTorch can build tensors directly from NumPy).


In [ ]:
# First column = label, rest = pixels
x = data.iloc[:, 1:]
y = data.iloc[:, 0]
x.shape, y.shape


In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
x_train.shape, x_test.shape, y_train.shape, y_test.shape


In [ ]:
# Normalize: pixel values 0-255 -> 0.0-1.0
x_train = x_train / 255.0
x_test  = x_test  / 255.0


In [ ]:
# Convert DataFrames -> NumPy arrays
x_train = x_train.to_numpy()
x_test  = x_test.to_numpy()
y_train = y_train.to_numpy()
y_test  = y_test.to_numpy()


## 4 · Custom Dataset & DataLoader

- `MNISTdataset` wraps the NumPy arrays as a PyTorch `Dataset`.
- `DataLoader` iterates the dataset in mini-batches of 32.
  - `shuffle=True` for training (so the model doesn't see data in label order)
  - `shuffle=False` for test (order doesn't matter for accuracy)


In [ ]:
class MNISTdataset(Dataset):
    def __init__(self, x, y):
        # x: float32 (neural nets compute in float32)
        # y: long (CrossEntropyLoss expects integer class indices)
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


In [ ]:
train_dataset = MNISTdataset(x_train, y_train)
test_dataset  = MNISTdataset(x_test,  y_test)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    pin_memory=torch.cuda.is_available(),
)


In [ ]:
# Sanity check: print the shape of the first batch
for batch_x, batch_y in train_loader:
    print('Batch X shape:', batch_x.shape)   # (32, 784)
    print('Batch Y shape:', batch_y.shape)   # (32,)
    break


## 5 · Model & training setup

A 6-layer feed-forward MLP with ReLU activations between layers.
Output layer produces 10 raw logits — `CrossEntropyLoss` applies softmax internally.


In [ ]:
class MnistNN(nn.Module):
    def __init__(self, n_feature, n_classes=10):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(n_feature, 512), nn.ReLU(),
            nn.Linear(512, 256),      nn.ReLU(),
            nn.Linear(256, 128),      nn.ReLU(),
            nn.Linear(128, 64),       nn.ReLU(),
            nn.Linear(64, 32),        nn.ReLU(),
            nn.Linear(32, n_classes),  # no activation: raw logits
        )

    def forward(self, x):
        return self.model(x)


In [ ]:
model = MnistNN(n_feature=x_train.shape[1], n_classes=10).to(device)
print(model)


In [ ]:
lr       = 3e-4
n_epochs = 10

loss_fn  = nn.CrossEntropyLoss()
optimizer = optim.Adam(params=model.parameters(), lr=lr)


In [ ]:
len(train_loader), len(test_loader)   # ~1500 train batches, ~375 test batches


### Per-epoch math

1 epoch ≈ 1,500 batches (48,000 / 32)  
10 epochs ≈ 15,000 weight updates total


## 6 · Training loop

The PyTorch training trinity (inside the inner loop):

1. `optimizer.zero_grad()` — clear gradients from the previous batch  
2. `loss.backward()`       — compute gradients via autograd  
3. `optimizer.step()`      — nudge weights against the gradient


In [ ]:
model.train()
train_losses = []

for epoch in range(n_epochs):
    tot_epoch_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        y_pred = model(batch_x)
        loss = loss_fn(y_pred, batch_y)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_epoch_loss += loss.item()

    avg_epoch_loss = tot_epoch_loss / len(train_loader)
    train_losses.append(avg_epoch_loss)
    print(f'Epoch [{epoch + 1} / {n_epochs}]: loss = {avg_epoch_loss:.4f}')


In [ ]:
# Plot the training-loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, n_epochs + 1), train_losses, marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Average training loss')
plt.title('Training Loss')
plt.grid(True, alpha=0.3)
plt.show()


## 7 · Evaluation

Switch the model to eval mode, disable gradient tracking, and count correct predictions.


In [ ]:
total_preds = len(test_dataset)
total_correct_preds = 0

model.eval()
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        y_pred = model(batch_X)
        pred_label = torch.argmax(y_pred, dim=1)

        batch_correct = (batch_y == pred_label).sum().item()
        total_correct_preds += batch_correct


In [ ]:
accuracy = total_correct_preds / total_preds
print(f'Test accuracy: {accuracy * 100:.2f}%')
print(f'Correct: {total_correct_preds} / {total_preds}')


## 8 · Save the trained model

Saving weights so the upcoming PyGame demo can load them without retraining.


In [ ]:
torch.save(model.state_dict(), 'mnist_mlp.pt')
print('Saved to mnist_mlp.pt')


---

### ✅ Pipeline complete

**Next up:** a PyGame drawing pad that loads `mnist_mlp.pt` and lets you sketch digits with the mouse.
See the project README for the roadmap.
